# Map sequencing reads to wells

**What it does.** Read the amplicon FASTQs and count which guide RNA landed in which well.

**When to use it.** For pooled CRISPR screens, once sequencing comes back. This is what turns a plate of images into a genotype-to-phenotype table.

**What you get.** A per-well barcode count table, plus QC on read quality and consensus.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.sequencing.generate_barecode_mapping`

```
generate_barecode_mapping(settings=None)
```

Turn a folder of pooled-screen FASTQ files into per-well sgRNA count tables usable by :func:`spacr.ml.perform_regression`.

In [ ]:
from spacr.sequencing import generate_barecode_mapping

## 3. Settings

`spacr.settings.get_map_barcodes_default_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_map_barcodes_default_settings

defaults = get_map_barcodes_default_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

Every key, with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version by `tools/build_notebook_settings.py`, so these are the real keys, the real defaults and the real descriptions. Re-run that tool after upgrading spaCR.

In [ ]:
# Generated by tools/build_notebook_settings.py — do not edit by hand.
# Values are this installed version's defaults; the text above each key is
# its live description. Edit values in place -- a key left at its default
# behaves exactly as if it were absent.

# spacr.sequencing.generate_barecode_mapping  (17 settings)
generate_barecode_mapping_settings = {
    # (int) - Number of FASTQ reads read into memory and handed to each
    # worker batch. Larger chunks cut per-batch overhead and make the
    # progress bar coarser but raise peak RAM per job; smaller chunks
    # stream more gently on low-memory machines. Also sets how many reads
    # are processed when test is True. Default 100000.
    'chunk_size': 100000,

    # (path) - CSV mapping column barcodes to well names; it must have
    # 'sequence' and 'name' columns. Reads are matched verbatim against it
    # with no reverse-complementing, so the sequences must be in the same
    # orientation as the reads - run barecodes_reverse_complement on the
    # file if they are not. Unmatched reads get NA for columnID. Default
    # the bundled spacr/resources/data/barcodes_column.csv; barcode QC
    # (sequencing_qc) instead defaults this key to empty, where the
    # reference is optional.
    'column_csv': str(__import__('importlib.resources', fromlist=['files']).files('spacr.resources.data').joinpath('barcodes_column.csv')),

    # (int) - complevel passed to the HDF5 store, 0-9. 0 disables
    # compression (fastest write, largest file); higher values shrink
    # annotated_reads.h5 at increasing CPU cost, and at the top of the
    # range saving can take longer than the barcode mapping itself.
    # Ignored when save_h5 is False. Default 5.
    'comp_level': 5,

    # (str) - PyTables compression library used when writing
    # annotated_reads.h5, passed to pandas HDFStore as complib: 'zlib',
    # 'lzo', 'bzip2' or 'blosc'. 'blosc' is far faster at similar file
    # size, 'bzip2' is smallest but slowest. Ignored entirely when save_h5
    # is False. Default 'zlib'.
    'comp_type': 'zlib',

    # (int) - Number of bases sliced out of each read starting at
    # offset_start relative to the target_sequence hit; this window is
    # what the regex is matched against. It must span the whole barcode
    # block (column + gRNA + row) or the regex stops matching and reads
    # are dropped; shorter reads are padded with 'N'. Default 89.
    'expected_end': 89,

    # (bool) - When a barcode does not match any entry in its reference
    # CSV, count it under its raw sequence instead of dropping it. Off,
    # the groupby silently discards every unmatched read, so a reference
    # with the wrong orientation produces a small clean table rather than
    # an obviously empty one. Turn it on to see how much of the run failed
    # to map. Default False.
    'fill_na': False,

    # (path) - CSV mapping gRNA barcode sequences to gRNA names; it must
    # have 'sequence' and 'name' columns. Reads are matched verbatim with
    # no reverse-complementing, so orientation must match the reads
    # (barecodes_reverse_complement flips a file). Rows whose gRNA does
    # not match are written as NA and dropped from the counts. Default:
    # the bundled spacr/resources/data/grna_barcodes.csv.
    'grna_csv': str(__import__('importlib.resources', fromlist=['files']).files('spacr.resources.data').joinpath('barcodes_grna.csv')),

    # (str) - Read-pairing strategy for barcode extraction: 'paired'
    # locates target_sequence in R1 and in the reverse complement of R2
    # and merges them base-by-base into a quality-weighted consensus;
    # 'single' scans one mate alone, chosen by single_direction. Paired
    # calls barcodes more accurately but discards any read whose anchor is
    # missing from either mate. Default 'paired'.
    'mode': 'paired',

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where -1
    # means every core. Raise it to shorten CPU-bound steps until RAM or
    # disk I/O saturates. Note the measure-and-crop pipeline overrides
    # your value with cpu_count()-4. Defaults vary by pipeline:
    # cpu_count()-4, -1, or None.
    'n_jobs': None,

    # (int) - Bases to shift from the start of the target_sequence match
    # to the start of the extracted window; negative values move upstream
    # to capture a barcode preceding the anchor. The start is clamped at
    # position 0, so an over-negative value silently shifts the reading
    # frame and the regex stops matching. Default -8.
    'offset_start': -8,

    # (str) - Regex applied with re.match to each extracted read window;
    # it must define the named groups columnID, grna and rowID, whose
    # captured sequences are looked up in the three barcode CSVs. Non-
    # matching reads are silently dropped, so a wrong group name or
    # barcode orientation yields zero counts. The default captures an 8 bp
    # column, 20-21 bp gRNA and 8 bp row barcode.
    'regex': '^(?P<columnID>.{8})TGCTG.*TAAAC(?P<grna>.{20,21})AACTT.*AGAAG(?P<rowID>.{8}).*',

    # (path) - CSV mapping row barcodes to well names; it must have
    # 'sequence' and 'name' columns. Reads are matched verbatim with no
    # reverse-complementing, so the sequences must be in the same
    # orientation as the reads - use barecodes_reverse_complement to flip
    # the file if needed. Unmatched reads get NA for rowID. Default: the
    # bundled spacr/resources/data/barcodes_row.csv.
    'row_csv': str(__import__('importlib.resources', fromlist=['files']).files('spacr.resources.data').joinpath('barcodes_row.csv')),

    # (bool) - Also write every annotated read (consensus sequence plus
    # its parsed row/column/gRNA barcodes and IDs) to annotated_reads.h5.
    # The per-well counts in unique_combinations.csv and qc.csv are
    # written either way, so set it False unless you need read-level data;
    # True produces a very large file and compression can dominate
    # runtime. Default True.
    'save_h5': True,

    # (str) - Which mate to scan when mode is 'single': 'R1' or 'R2'. The
    # chosen file is read as-is with no reverse-complementing, so
    # selecting 'R2' means target_sequence and regex must be written in R2
    # orientation or nothing will match. Ignored when mode is 'paired'.
    # Default 'R1'.
    'single_direction': 'R1',

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run. No usable default: the settings factories
    # fill a placeholder ('path' or '/path/to/src'), so this must be
    # supplied.
    'src': 'path',

    # (str) - Constant vector sequence used as the anchor: every read is
    # scanned for an exact match and the barcode window is then sliced
    # relative to that hit using offset_start and expected_end. Reads
    # without an exact match are skipped entirely, so it must be error-
    # free and given in the orientation of the read being scanned. Default
    # 'TGCTGTTTCCAGCATAGCTCTTAAAC'.
    'target_sequence': 'TGCTGTTTCCAGCATAGCTCTTAAAC',

    # (bool) - In classifier training, run the held-out evaluation pass
    # (combine with train, or use alone to score an existing model). In
    # the sequencing barcode mapper it means something different: process
    # only the first read chunk and print a preview, so you can sanity-
    # check the regex and barcode CSVs in seconds. Default False.
    'test': False,
}

# spacr.ml.perform_regression  (64 settings)
perform_regression_settings = {
    # (str) - How per-object scores are collapsed to one value per well
    # before regression: 'mean', 'median', 'quantile' (75th percentile),
    # or None to skip aggregation and regress on individual objects.
    # Median resists a handful of extreme cells; None keeps power but
    # ignores within-well correlation. Forced to a per-well sum for
    # poisson and to None for quantile. Default 'mean'.
    'agg_type': 'mean',

    # (float) - Regularisation strength for the penalised models only: the
    # L1 penalty for 'lasso', the L2 penalty for 'ridge', the combined
    # penalty for 'elasticnet' and the inverse margin for 'hinge'. Larger
    # values shrink more coefficients toward zero; set it to 'auto' or
    # None to choose it by 5-fold cross-validation, which is usually what
    # you want because the default 1 shrinks a fraction-scale design to
    # nothing. Every other regression type refuses a non-default alpha
    # rather than ignoring it, and it is no longer the quantile - see the
    # quantile setting. Default 1.
    'alpha': 1,

    # (str) - 'regression' fits the selected simultaneous model.
    # 'guide_permutation' tests each guide as a plate-adjusted marginal
    # association using blocked Freedman--Lane permutations and then
    # corrects the requested support family. Default 'regression'.
    'analysis_mode': 'regression',

    # (str) - Metadata column that identifies independent acquisition
    # batches, normally 'plateID'. Every analyzed row must have a value
    # and at least batch_min_samples rows must occur in each batch. Use an
    # acquisition date or instrument ID only if that is the nuisance
    # source you intend to remove. Default 'plateID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_column': 'plateID',

    # (bool) - True corrects only the additive batch shift and leaves each
    # batch's scale alone. Use it when the plates differ in level but not
    # in spread, or when a batch has too few rows for a stable variance
    # estimate. False (the default) corrects both location and scale,
    # which is standard ComBat. Ignored by every method other than combat.
    # API: spacr.batch_correction.correct_batch_effects.
    'batch_combat_mean_only': False,

    # (str or None) - Metadata column containing reference-control labels
    # for control_center, normally 'columnID' for plate controls. It is
    # ignored by center, zscore, robust_zscore, and none. Blank follows
    # col_to_compare in Image UMAP or location_column in Classify (ML);
    # regression defaults to 'columnID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_control_column': 'columnID',

    # (str, number, list or None) - Reference/negative-control value(s) in
    # batch_control_column used by control_center. Each plate needs at
    # least batch_min_samples matching rows. Image UMAP falls back to neg
    # and Classify (ML) to negative_control when this field is blank;
    # regression requires an explicit value. Default varies by module.
    # API: spacr.batch_correction.correct_batch_effects.
    'batch_control_values': None,

    # (str) - Plate/batch correction applied before Image UMAP, ML screen
    # classification or phenotype regression. 'none' leaves measurements
    # alone; 'center' removes each plate's mean shift; 'zscore' aligns
    # plate means and variances; 'robust_zscore' uses median/MAD and
    # tolerates outliers; 'combat' models the batch effect while
    # protecting the terms named in batch_covariate_column. Correct when
    # plates were stained or imaged separately; leave off when they were
    # not, since every method removes real signal that happens to align
    # with plate. See spacr.batch_correction.correct_batch_effects.
    # Default 'none'.
    'batch_correction': 'none',

    # (str, list or None) - Metadata column(s) naming the biology combat
    # must PROTECT, e.g. 'condition' or 'condition,timepoint'. combat
    # estimates the batch effect from the residuals after these terms, so
    # anything NOT listed is treated as noise and removed along with the
    # plate effect. Leave your treatment out of this list and combat will
    # quietly delete the effect you are measuring. See
    # spacr.batch_correction.correct_batch_effects. Default None.
    'batch_covariate_column': None,

    # (int) - Minimum number of rows required in every batch, and minimum
    # matching reference controls per batch for control_center. Correction
    # stops with an actionable error below this threshold because a one-
    # or two-object plate estimate is unstable. Default 3. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_min_samples': 3,

    # (str) - Policy when control_center cannot find enough reference
    # controls on a plate: 'error' stops rather than silently mixing
    # corrected and raw plates; 'skip' leaves that plate unchanged and
    # records a warning. Default 'error'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_missing_control': 'error',

    # (list or None) - Wells whose parasites carry no pre-permeabilisation
    # stain, giving the honest negative distribution the cut should sit
    # above -- better evidence than any automatic method. Name a column
    # ('c12'), a row ('r1'), a well ('r1_c12') or a full plate key. These
    # wells are dropped from every efficiency, since a staining control is
    # not an experimental condition. None runs the automatic per-field
    # method instead. NOTE the screen regression reads this same key for a
    # different job, where it must be a list matching filter_value.
    # Default None.
    'control_wells': ['c1', 'c2', 'c3'],

    # (list) - gRNA identifiers treated as non-targeting controls in the
    # regression. Their coefficients are tagged 'control', and their
    # spread sets reg_threshold = mean + threshold_multiplier x (std or
    # var, per threshold_method) - the effect-size cut-off drawn on the
    # volcano plot. A noisier or wider control set raises that bar. None
    # skips the threshold. Default the built-in list of 30 non-targeting
    # IDs, '000000_1' to '000000_32' (without _2 and _7).
    'controls': ['000000_1', '000000_10', '000000_11', '000000_12', '000000_13', '000000_14', '000000_15', '000000_16', '000000_17', '000000_18', '000000_19', '000000_20', '000000_21', '000000_22', '000000_23', '000000_24', '000000_25', '000000_26', '000000_27', '000000_28', '000000_29', '000000_3', '000000_30', '000000_31', '000000_32', '000000_4', '000000_5', '000000_6', '000000_8', '000000_9'],

    # (str or list) - CSV(s) of per-well gRNA read counts from the
    # sequencing step (unique_combinations.csv); each must contain grna,
    # count, rowID and columnID columns or the run raises ValueError.
    # These are the regression's independent variable. Pass one path per
    # plate, position-aligned with plates_count; results are written under
    # the first file's folder. Default 'list of paths', a placeholder that
    # must be replaced; the barcode QC module defaults this key to 'path
    # to unique_combinations.csv'.
    'count_data': 'list of paths',

    # (str) - Heteroscedasticity-robust covariance estimator passed to the
    # likelihood fits: 'HC0', 'HC1', 'HC2' or 'HC3', or None for classical
    # non-robust errors. It changes standard errors and p-values only,
    # never the coefficients; reach for 'HC3' when residual variance grows
    # with well cell count. Applies to regression_type 'ols', 'wls',
    # 'glm', 'poisson', 'quasi_binomial', 'logit' and 'probit'; the
    # penalised, robust and quantile fits have no such estimator and
    # refuse it rather than quietly reporting ordinary errors under a
    # robust label. Default None.
    'cov_type': None,

    # (str) - Name of the column in score_data that is modelled as the
    # response, e.g. 'pred'/'predictions' from the ML scoring step or a
    # measured feature such as 'pathogen_nucleus_shortest_distance'. It is
    # aggregated per well by agg_type and then optionally transformed. The
    # run aborts if the column is absent from the score CSV. Default
    # 'pred'.
    'dependent_variable': 'pred',

    # (float) - Family-level rejection threshold for adjusted P values in
    # guide_permutation mode. Must be between 0 and 1. Default 0.05.
    'fdr_alpha': 0.05,

    # (str) - Metadata column used to drop control wells before
    # regression: every row whose value appears in filter_value is removed
    # from both the score data and the read counts. Use 'columnID'
    # (default) when controls sit in plate columns, 'rowID' when they sit
    # in rows. In annotate_filter_vision it instead names the score column
    # thresholded by upper_threshold/lower_threshold.
    'filter_column': 'columnID',

    # (list) - Values of filter_column whose rows are removed - not kept -
    # before regression, normally the control columns; default
    # ['c1','c2','c3']. Dropping them stops control wells from dominating
    # the gene and gRNA fits. Only list values take effect: a bare string
    # is silently ignored and nothing is filtered.
    'filter_value': ['c1', 'c2', 'c3'],

    # (float) - Minimum relative abundance, 0-1, that a gRNA must reach
    # within a well's total read count to be kept. Raising it strips low-
    # abundance and bleed-through gRNAs and lowers the mean gRNAs per
    # well; set it too high and every row is removed and the run errors
    # out. Leave None to auto-pick the cutoff giving target_unique_count
    # gRNAs per well. Default None.
    'fraction_threshold': None,

    # (int or list) - Minimum numbers of independent wells containing a
    # guide. A list writes one sensitivity-analysis table and volcano plot
    # per threshold; P values are computed once and correction is repeated
    # within each eligible family. Default [1, 2, 3, 4].
    'guide_min_wells': [1, 2, 3, 4],

    # (list) - Additional measured well-level covariates to residualize
    # from both phenotype and guide fraction before testing. Do not put
    # post-treatment outcomes here. Default [].
    'guide_nuisance_columns': [],

    # (int) - Number of permutation outcomes evaluated together. Lower
    # this if memory is tight; it does not change the result. Default 500.
    'guide_permutation_batch_size': 500,

    # (str) - Column defining exchangeability blocks for permutations,
    # normally plateID. Residuals are never shuffled between its levels.
    # Default 'plateID'.
    'guide_permutation_block': 'plateID',

    # (bool) - Write PDF and PNG volcano plots for every requested
    # guide_min_wells support family. Disable it for table-only batch
    # runs; inference is unchanged. Default True.
    'guide_permutation_plot': True,

    # (int) - Random seed for reproducible residual permutations. Keep it
    # fixed to reproduce exact empirical P values; change it to check
    # Monte Carlo sensitivity. Default 0.
    'guide_permutation_seed': 0,

    # (int) - Number of plate-blocked Freedman--Lane residual permutations
    # used for empirical two-sided guide P values. More permutations
    # improve tail resolution but take longer. Default 200000.
    'guide_permutations': 200000,

    # (float) - A guide counts as present in a well only when its fraction
    # is above this value. The effect still uses the unthresholded
    # fraction. Default 0.0.
    'guide_presence_threshold': 0.0,

    # (int or None) - Which guide_min_wells family supplies
    # results_significant.csv and the returned significant table. Default
    # None chooses the smallest requested threshold.
    'guide_primary_min_wells': None,

    # (int) - Number of bootstrap resamples behind the hinge p-values. A
    # support vector machine has no likelihood and so no Wald test; spaCR
    # refits it on this many resamples of the wells and compares each
    # coefficient to its bootstrap standard deviation. Treat the result as
    # a stability statistic, not a hypothesis test. Higher is steadier and
    # linearly slower; below about 50 the standard deviations are too
    # noisy to rank on. Default 200.
    'hinge_n_boot': 200,

    # (float) - Response value above which a well counts as positive for
    # the hinge (linear SVM) fit. Leave it None when the response is
    # already binary, in which case the two values it holds become the two
    # classes. spaCR refuses a continuous response with no threshold
    # rather than splitting it at the mean or median, because a cut chosen
    # by the software decides the hypothesis being tested. Read only by
    # regression_type 'hinge'. Default None.
    'hinge_threshold': None,

    # (float) - Where Huber's loss switches from squared to linear, in
    # units of the estimated residual scale, for the robust fits. Smaller
    # values downweight more wells and resist heavier contamination;
    # larger values approach ordinary least squares. The default 1.345
    # gives 95 percent of the efficiency of OLS when the residuals really
    # are normal. Read only by regression_type 'rlm' and 'huber'. Default
    # 1.345.
    'huber_t': 1.345,

    # (bool or int) - Flip the response before it is aggregated per well,
    # for scores whose useful direction is downward. False or 0 leaves it
    # as measured, True or 1 uses 1 - x (right for a probability, so a low
    # infection score becomes a high phenotype), and -1 uses 1 / x (right
    # for a distance or a count). Any other value raises ValueError in
    # process_scores. It changes the sign of every coefficient and
    # therefore which side of the volcano your hits land on. Default
    # False.
    'invert_dependent_variable': False,

    # (float) - How the elastic-net penalty is split between L1 and L2:
    # 1.0 is a pure lasso (sparse, picks one gRNA out of a correlated
    # group), 0.0 is a pure ridge (dense, shares the effect across the
    # group), and values between keep some of both. Use 0.5 when
    # correlated gRNAs of the same gene should be selected together rather
    # than arbitrarily. Read only by regression_type 'elasticnet'. Default
    # 0.5.
    'l1_ratio': 0.5,

    # (int) - Number of bootstrap resamples used to rank lasso and
    # elastic-net hits by how often each gRNA survives the penalty. These
    # models have no valid p-values, so selection frequency replaces the
    # significance test entirely. Higher is steadier and linearly slower;
    # the cost is one full penalised fit per resample, doubled when alpha
    # is 'auto' because each resample cross-validates. Default 200.
    'lasso_n_boot': 200,

    # (float) - Minimum bootstrap selection frequency, between 0 and 1,
    # for a lasso or elastic-net coefficient to be called a hit. 0.6 means
    # the gRNA kept a non-zero coefficient in at least three fifths of the
    # resamples. Raise it for a shorter, harder-to-argue-with list;
    # lowering it below about 0.5 admits terms the penalty drops as often
    # as it keeps. Default 0.6.
    'lasso_selection_threshold': 0.6,

    # (bool) - Put the x-axis on a log10 scale; for line graphs the x
    # column is log10-transformed instead of the axis being rescaled.
    # Enable it when x spans orders of magnitude - gRNA fraction
    # thresholds, count distributions - so the low end is not squashed
    # against the axis. Values at or below zero cannot be shown. Default
    # False.
    'log_x': False,

    # (bool) - Put the y-axis on a log10 scale; for line graphs the y
    # column is log10-transformed instead of the axis being rescaled.
    # Enable it when the measured values span orders of magnitude or a few
    # large wells compress everything else toward the baseline. Values at
    # or below zero cannot be shown. Default False.
    'log_y': False,

    # (float or None) - Fraction of failed items above which the run
    # aborts rather than finishing and reporting. 0.2 means 'stop once
    # more than a fifth of the fields have failed', on the grounds that
    # whatever is left is no longer the experiment. The ledger is stamped
    # into the artifact before the abort, so the evidence survives. None
    # (the default) never aborts on rate alone - every failure is still
    # counted and reported, and the artifact is still marked partial.
    # Default None.
    'max_failure_rate': None,

    # (list) - Gene-annotation CSVs, each with a 'Gene ID' column, that
    # are joined onto the regression results by gene, writing an extra
    # results CSV per file. These are gene tables, not plate/well
    # metadata. When toxo is True the order matters: index 0 is read as
    # the ME49 transcription table and index 1 as the GT1 phenotype table.
    # Default [].
    'metadata_files': [],

    # (int) - Wells with fewer than this many scored objects are dropped
    # before regression. Raising it removes noisy, sparsely imaged wells
    # at the cost of statistical power. Leave None and spaCR simulates the
    # count at which a well's mean score stabilises within tolerance and
    # uses that value. Default None.
    'min_cell_count': None,

    # (int) - Observation count a significant hit must strictly exceed to
    # appear in results_significant_filtered.csv: gRNA hits need n_grna >
    # min_n, gene hits need n_gene > min_n. The unfiltered hit list is
    # still written alongside it. Raise it to drop hits resting on one or
    # two wells. Default 0, which filters nothing.
    'min_n': 0,

    # (str) - Correction applied within each outcome/support family:
    # fdr_bh (Benjamini--Hochberg, default), fdr_by, bonferroni, holm, or
    # none. Stricter family-wise methods generally call fewer guides.
    'multiple_testing_method': 'fdr_bh',

    # (str) - Identifier of the negative-control class. In ML screening it
    # is the value in location_column (e.g. 'c1') whose objects are
    # labelled class 0 for training; in gRNA regression it is a gene/gRNA
    # ID substring (e.g. '233460') matched against coefficient names to
    # tag them 'nc' in the results and volcano plot. Defaults 'c1' and
    # '233460' respectively.
    'negative_control': '233460',

    # (bool) - After building the regression table, drop gRNAs whose well
    # count falls outside 1.5x the 5th-95th percentile spread, then
    # recompute the per-gRNA tables. This removes gRNAs present in
    # implausibly few or many wells that would otherwise dominate
    # coefficients; disable it if your library is deliberately uneven.
    # Default True.
    'outlier_detection': True,

    # (str) - Plate name stamped onto count and score rows that carry no
    # plate of their own, and used as the first field of the
    # plate_row_column key that joins the two tables. It is ignored with a
    # warning when the input already contains more than one distinct
    # plate, so it matters only for single-plate inputs. Default 'plate1'.
    'plateID': 'plate1',

    # (str) - Identifier of the positive-control class. In ML screening it
    # is the value in location_column (e.g. 'c2') whose objects are
    # labelled class 1 for training; in gRNA regression it is a gene/gRNA
    # ID substring (e.g. '239740') matched against coefficient names to
    # tag them 'pc' in the results and volcano plot. Defaults 'c2' and
    # '239740' respectively.
    'positive_control': '239740',

    # (float) - Which quantile of the response quantile regression fits,
    # strictly inside 0 and 1: 0.5 is the median (robust to outlier
    # wells), 0.9 asks which gRNAs move the top of the distribution rather
    # than its centre. Aggregation is turned off automatically so the
    # quantile is taken over cells, not over well means. Read only by
    # regression_type 'quantile'; it replaced the old overload of alpha.
    # Default 0.5.
    'quantile': 0.5,

    # (bool) - Fit plate, row and column as random effects instead of
    # fixed ones: True overrides regression_type to 'mixed' and fits a
    # MixedLM grouped by plateID with rowID and columnID variance
    # components, dropping them from the fixed-effect formula. Use it when
    # edge or row artefacts differ between plates; it is slower and may
    # fail to converge. Default False.
    'random_row_column_effects': False,

    # (str) - Choose by what the per-well response IS. Continuous: 'ols',
    # 'wls' (weight by cell count), 'rlm'/'huber' (robust), 'quantile'. A
    # FRACTION: 'logit', 'probit', 'beta', 'quasi_binomial' (over-
    # dispersed). Counts: 'poisson'. 'glm' picks the family. 'mixed' adds
    # well or plate as a random effect. Many correlated predictors:
    # 'ridge' shrinks all, 'lasso' zeroes some, 'elasticnet' both,
    # 'horseshoe' shrinks hard but spares large effects. 'hinge' for a
    # separable two-class response. ols on a bounded response predicts
    # values outside it. Default 'ols'.
    'regression_type': 'ols',

    # (str) - Which column of the per-object score CSV
    # minimum_cell_simulation resamples when it works out how many objects
    # a well needs before its mean stops moving. It must name the same
    # measurement as dependent_variable, or the simulated min_cell_count
    # describes a different quantity than the one the regression fits and
    # wells are kept or dropped on the wrong evidence; the regression
    # defaults therefore follow dependent_variable. In the interpret-
    # vision-model helper the same key names the CNN score column instead,
    # default 'cv_predictions'.
    'score_column': 'pred',

    # (str or list) - CSV(s) of per-object or per-well phenotype scores
    # (typically the output of generate_ml_scores) supplying the
    # regression's dependent variable; the column named by
    # dependent_variable must be present or the run raises ValueError.
    # Pass one path per plate, position-aligned with plates_score; the
    # first file's name becomes the results subfolder. Default 'list of
    # paths'.
    'score_data': 'list of paths',

    # (str) - Intended to fix the axis limits of split/faceted plots as
    # [xmin, xmax, ymin, ymax], but nothing reads
    # settings['split_axis_lims']; those plots always autoscale. Use the
    # per-plot x_lim / y_lim settings where the plotting function exposes
    # them. Kept only so old settings CSVs still load. Default empty.
    'split_axis_lims': '',

    # (bool or None) - What happens when a step hits a problem it could
    # survive. OFF records the failure in the run ledger and the end-of-
    # run summary and carries on with the items that worked. ON raises
    # immediately on a setup or configuration error -- an unreadable path,
    # a missing column, a database that will not open -- so a batch stops
    # at the first sign its inputs are wrong instead of producing a
    # plausible partial result. Per-item failures such as one corrupt
    # image are survived either way. None defers to $SPACR_STRICT_ERRORS,
    # which is how a cluster sets it for a whole batch. Default None.
    'strict_errors': None,

    # (int) - Desired mean number of distinct gRNAs per well. spaCR sweeps
    # 1000 read-fraction thresholds, picks the one whose per-well mean
    # unique gRNA count lands closest to this number, then discards every
    # gRNA call below that fraction. Lower it for a stricter, cleaner well
    # assignment; raise it to keep more gRNAs per well. Default 5.
    'target_unique_count': 5,

    # (str) - How the spread of the control-gRNA regression coefficients
    # is measured when the hit-calling threshold is built: 'std' or
    # 'standard_deveation' uses the standard deviation, 'var' or
    # 'variance' uses the variance (much wider once the spread exceeds 1).
    # Any other value raises an error. Only used when 'controls' is set.
    # Default 'std'.
    'threshold_method': 'std',

    # (float) - How many units of control-coefficient spread are added to
    # the mean control coefficient to form the regression hit threshold:
    # reg_threshold = mean(control coefficients) + multiplier * spread,
    # where spread comes from threshold_method. Larger values place the
    # threshold further out in the control distribution. Only used when
    # 'controls' is set. Default 3.
    'threshold_multiplier': 3,

    # (int or float) - How close a subsampled well mean has to be to the
    # full-well mean before minimum_cell_simulation calls that sample size
    # sufficient, which is what sets min_cell_count when you leave it
    # None. An int is read as a percentage (2 means 2%), a float as a
    # fraction (0.02 means the same); anything else raises ValueError.
    # Tighten it toward 0.01 to demand more cells per well and drop more
    # wells, loosen it to 0.05 to keep sparse wells at the cost of noisier
    # per-well scores. Default 0.02.
    'tolerance': 0.02,

    # (bool) - Merge the regression hits with the bundled Toxoplasma
    # metadata (LOPIT/TAGM localisations in resources/data/lopit.csv) and,
    # from that, draw the volcano plot plus GT1 phenotype and ME49
    # transcription heatmaps read from metadata_files. Turn it off for
    # non-Toxoplasma screens - doing so also disables the volcano plot
    # entirely. Default True.
    'toxo': True,

    # (str) - Optional transform applied to the aggregated per-well
    # response before fitting: 'log' (log1p), 'sqrt', 'square', or None
    # for none. Reach for it when the response is skewed and the normality
    # check fails; the fit then reports coefficients for the transformed
    # column, named '<transform>_<dependent_variable>'. Default None.
    'transform': None,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table, the channel and model choices per object
    # type, per-table row counts, and how many objects survive each
    # filter. It only adds console output, so turn it on when object
    # counts come out unexpected and you need to see which stage removed
    # them. The default differs per pipeline -- True for mask, UMAP,
    # screen analysis, barcode mapping and Cellpose training; False for
    # measure, the plotting helpers and regression.
    'verbose': False,

    # (str) - Which coefficient table the volcano plot is drawn from:
    # 'gene' (default) plots per-gene coefficients, 'grna' per-gRNA, 'all'
    # the full merged table; any other value skips the plot. Points are
    # coloured by TAGM/LOPIT localisation, and the gene list it returns
    # drives the phenotype and transcription plots. Only takes effect when
    # toxo is True.
    'volcano': 'gene',

    # (list) - Two-element [min, max] limits on the coefficient (x) axis
    # of the Toxoplasma volcano plot produced by the regression pipeline
    # when toxo mode is on. Narrow it to zoom in on hits clustered near
    # zero, widen it to keep large-effect genes on the plot. Leaving it
    # None falls back to [-0.5, 0.5], not auto-scaling. Default None.
    'x_lim': None,

    # (list or None) - Limits of the -log10(p) axis of the Toxoplasma
    # volcano plot. None auto-scales to the data; [low, high] fixes the
    # axis so several plates can be compared at the same scale; [[low1,
    # high1], [low2, high2]] draws a broken axis with the gap between the
    # two ranges removed, which keeps a handful of extremely significant
    # genes on the plot without flattening everything else. Any other
    # shape raises ValueError. Default None.
    'y_lims': None,
}

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
generate_barecode_mapping(settings)

## Where the output went

A per-well barcode count table, plus QC on read quality and consensus.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.